# OMA (Orthologous MAtrix) — Data Ingestion

**OMA** (Orthologous MAtrix) is a method and database for the inference of orthologs among complete genomes. Developed at ETH Zurich and SIB, OMA is one of the ELIXIR Core Data Resources and currently covers **>2,800 genomes** spanning all three domains of life, making it one of the most comprehensive orthology resources available.

Unlike bidirectional-best-hit (BBH) approaches, OMA performs **all-against-all Smith-Waterman** alignments across the full genome set, detects evolutionary **distances with maximum-likelihood** estimation, and then groups proteins into three complementary structures:

| Structure | Definition | Use case |
|---|---|---|
| **OMA Pairs** | Pairs of genes inferred to have descended from a single ancestral gene via speciation (strict orthologs). | Function transfer, 1:1 mappings between species. |
| **OMA Groups** | Cliques of pairwise orthologs — every member is a pairwise ortholog of every other member. Conservative, all-1:1. | Species-tree inference, deep conservation studies. |
| **HOG (Hierarchical Orthologous Group)** | Set of genes descended from a single ancestral gene at a given taxonomic level. Nested along the species tree. | Comparative genomics at variable taxonomic depths, paralog analysis. |

**Key biological concepts:**

- **Orthologs** share a common ancestor via a speciation event (vertical inheritance).
- **Paralogs** share a common ancestor via a duplication event (within-lineage expansion).
- Under the **ortholog conjecture**, orthologs are more likely than paralogs to retain ancestral function — essential for reliable cross-species functional annotation transfer.

**REST API base:** `https://omabrowser.org/api/`

**Bulk downloads:** `https://omabrowser.org/All/` (OrthoXML, HDF5, TSV flat files)

**Reference:** Altenhoff A. M. *et al.* (2024), *Nucleic Acids Research*, OMA orthology in 2024. https://doi.org/10.1093/nar/gkad1020

In [ ]:
import requests
import time
from pathlib import Path

import polars as pl

# TODO

* [x] **Ingest data**
    * [x] Connect to the OMA REST API and confirm access via the API root
    * [x] Resolve a UniProt accession (human TP53, `P04637`) to an OMA protein entry
    * [x] Fetch orthologs for the target protein
    * [x] Fetch the Hierarchical Orthologous Group (HOG) for the same protein
    * [x] Parse responses into Polars DataFrames
    * [x] Save data to `data/` with caching
* [ ] **Explore and clean**
    * [ ] Summarise ortholog counts by relationship type (1:1, 1:many, many:many)
    * [ ] Inspect taxonomic distribution of orthologs
    * [ ] Compare HOG membership across different taxonomic levels
    * [ ] Handle ambiguous species assignments and deprecated entries
* [ ] **Analysis**
    * [ ] Compare OMA pairs vs. HOG membership for the same seed protein
    * [ ] Measure ortholog conservation along the species tree
    * [ ] Cross-reference OMA orthologs with UniProt functional annotations
    * [ ] Investigate lineage-specific gene duplications within the HOG
* [ ] **Visualization**
    * [ ] Phylogenetic tree of HOG members highlighting duplication events
    * [ ] Heatmap of ortholog presence/absence across clades
    * [ ] Sunburst / icicle chart of HOG hierarchy levels
* [ ] **Statistical analysis**
    * [ ] Discuss uncertainty in orthology inference (alignment confidence, distance estimation)
    * [ ] Hypothesis framework for the ortholog conjecture (function conservation)
    * [ ] Multiple hypothesis correction when testing conservation across thousands of orthologs
    * [ ] Error propagation from pairwise distances to hierarchical group assignment

## 1. Ingest Data

### 1.1 Connect to the OMA REST API

The OMA REST API is documented at https://omabrowser.org/api/docs and serves JSON by default. All endpoints are rooted at `https://omabrowser.org/api/`. The API is unauthenticated but subject to rate limiting — we add a small delay between calls to be polite.

In [ ]:
OMA_API_BASE = "https://omabrowser.org/api"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)  # create data/ if it does not already exist

# The OMA API returns JSON by default; we still send an explicit Accept
# header so intermediate proxies do not negotiate to HTML.
HEADERS = {"Accept": "application/json", "User-Agent": "elixir-of-life/oma-demo"}


def oma_get(endpoint: str, params: dict | None = None) -> dict | list:
    """
    Send a GET request to the OMA REST API and return the parsed JSON body.

    Parameters
    ----------
    endpoint : str
        Path relative to :data:`OMA_API_BASE`, e.g. ``"protein/P04637/"``.
        A leading slash is optional. A trailing slash is recommended by the
        OMA docs and is added automatically if missing.
    params : dict, optional
        Additional query-string parameters (e.g. ``{"per_page": 1000}``).

    Returns
    -------
    dict or list
        Parsed JSON response. Collection endpoints return a list (or a
        paginated object), detail endpoints return a dict.

    Raises
    ------
    requests.HTTPError
        If the server returns a non-2xx status code.
    """
    endpoint = endpoint.lstrip("/")
    if not endpoint.endswith("/") and "?" not in endpoint:
        endpoint += "/"  # OMA prefers trailing-slash URLs
    url = f"{OMA_API_BASE}/{endpoint}"

    resp = requests.get(url, headers=HEADERS, params=params or {}, timeout=60)
    resp.raise_for_status()
    time.sleep(0.3)  # polite delay — SIB fair-use policy
    return resp.json()


# ── Connectivity check: the API root advertises all top-level resources ─────
api_root = oma_get("")  # GET https://omabrowser.org/api/

print("OMA REST API — root index")
print("=" * 60)
if isinstance(api_root, dict):
    # The root returns a mapping of resource-name -> URL
    for name, url in list(api_root.items())[:15]:
        print(f"  {name:<25} {url}")
    print(f"  ... ({len(api_root)} resources advertised)")
else:
    # Fallback: print whatever was returned
    print(api_root)

### 1.2 Resolve a UniProt Accession to an OMA Protein

We use human **TP53** (UniProt accession `P04637`) as our seed gene. The OMA protein endpoint accepts a number of identifier types — UniProt accession, Ensembl gene/protein/transcript ID, EntrezGene ID, or OMA's internal 5-letter species code + numeric ID (e.g. `HUMAN00012`). The endpoint transparently resolves cross-references.

In [ ]:
# Seed gene: human tumour-suppressor TP53 (guardian of the genome)
SEED_UNIPROT = "P04637"
SEED_CACHE = DATA_DIR / f"protein_{SEED_UNIPROT}.json"


def fetch_oma_protein(query_id: str, cache_path: Path) -> dict:
    """
    Fetch an OMA protein entry by cross-reference ID, with on-disk caching.

    Parameters
    ----------
    query_id : str
        Any identifier accepted by the ``/protein/{id}/`` endpoint —
        a UniProt accession, Ensembl ID, Entrez ID, or OMA entry name.
    cache_path : Path
        Local JSON file to write/read. If it exists, it is reused verbatim.

    Returns
    -------
    dict
        The protein record, including the canonical ``omaid`` (e.g.
        ``"HUMAN02546"``), ``entry_nr``, sequence, taxonomic data, and
        hyperlinks to orthologs / HOG / OMA-group resources.
    """
    if cache_path.exists():
        # Use Polars' helper to parse JSON — avoids an explicit json import
        import json
        print(f"Cache hit: {cache_path}")
        return json.loads(cache_path.read_text())

    record = oma_get(f"protein/{query_id}")
    # Persist the raw response so downstream notebooks can re-load without
    # hitting the API again.
    import json
    cache_path.write_text(json.dumps(record, indent=2))
    print(f"Cached to: {cache_path}")
    return record


tp53 = fetch_oma_protein(SEED_UNIPROT, SEED_CACHE)

# Summarise the most useful fields; OMA returns many, but a few are canonical
summary_fields = [
    "entry_nr", "omaid", "canonicalid", "sequence_md5",
    "species", "taxon_id", "sequence_length",
    "oma_group", "oma_hog_id", "roothog_id",
]
print(f"OMA entry for UniProt {SEED_UNIPROT}")
print("=" * 60)
for key in summary_fields:
    val = tp53.get(key)
    # "species" is itself a nested dict with the scientific name inside
    if isinstance(val, dict):
        val = val.get("species") or val.get("scientific_name") or val
    print(f"  {key:<20} {val}")

### 1.3 Fetch Orthologs for the Seed Protein

The `/protein/{id}/orthologs/` endpoint returns **OMA pairwise orthologs** — proteins inferred to descend from a single ancestral gene via speciation. Each entry carries a `rel_type` field indicating whether the relationship is **1:1**, **1:n**, **m:1**, or **m:n** (many-to-many orthology arising from lineage-specific duplications after the speciation event).

In [ ]:
import json

ORTH_CACHE = DATA_DIR / f"orthologs_{SEED_UNIPROT}.json"


def fetch_orthologs(query_id: str, cache_path: Path) -> list[dict]:
    """
    Fetch the list of OMA pairwise orthologs for a seed protein.

    The endpoint is ``/protein/{id}/orthologs/``. Results are returned as a
    JSON array; OMA orthologs of highly-conserved genes can number in the
    thousands, so the response is cached verbatim on first retrieval.

    Parameters
    ----------
    query_id : str
        Seed protein identifier (UniProt, Ensembl, Entrez, or OMA entry name).
    cache_path : Path
        Local JSON file used to memoise the call.

    Returns
    -------
    list of dict
        One record per ortholog; each has keys including ``omaid``,
        ``canonicalid``, ``species``, ``rel_type``, ``distance``, and
        ``score`` (Smith-Waterman alignment score).
    """
    if cache_path.exists():
        print(f"Cache hit: {cache_path}")
        return json.loads(cache_path.read_text())

    # Request a large page so we collect the full set in one round-trip
    orthologs = oma_get(f"protein/{query_id}/orthologs", params={"per_page": 10_000})
    cache_path.write_text(json.dumps(orthologs, indent=2))
    print(f"Cached {len(orthologs)} orthologs to: {cache_path}")
    return orthologs


orthologs_raw = fetch_orthologs(SEED_UNIPROT, ORTH_CACHE)
print(f"Retrieved {len(orthologs_raw):,} orthologs for {SEED_UNIPROT}")

# ── Parse into a Polars DataFrame ───────────────────────────────────────────
# Each ortholog record has a nested "species" dict; we flatten the fields
# we care about into top-level columns.
def _orthologs_to_frame(records: list[dict]) -> pl.DataFrame:
    """
    Flatten the raw OMA ortholog records into a tabular Polars DataFrame.

    Parameters
    ----------
    records : list of dict
        Output of :func:`fetch_orthologs`.

    Returns
    -------
    pl.DataFrame
        Columns: ``omaid``, ``canonicalid``, ``entry_nr``, ``rel_type``,
        ``distance``, ``score``, ``species``, ``taxon_id``,
        ``sequence_length``.
    """
    rows = []
    for r in records:
        sp = r.get("species") or {}  # nested species metadata
        rows.append({
            "omaid":           r.get("omaid"),
            "canonicalid":     r.get("canonicalid"),
            "entry_nr":        r.get("entry_nr"),
            # Relationship cardinality (1:1, 1:n, m:1, m:n)
            "rel_type":        r.get("rel_type"),
            # Maximum-likelihood evolutionary distance (PAM units)
            "distance":        r.get("distance"),
            # Smith-Waterman alignment score
            "score":           r.get("score"),
            "species":         sp.get("species") if isinstance(sp, dict) else None,
            "taxon_id":        sp.get("taxon_id") if isinstance(sp, dict) else None,
            "sequence_length": r.get("sequence_length"),
        })

    return pl.DataFrame(rows).with_columns([
        # Cast the relationship label — only ~4 distinct values
        pl.col("rel_type").cast(pl.Categorical),
        # Numeric columns arrive as Python floats/ints; cast defensively
        pl.col("distance").cast(pl.Float64, strict=False),
        pl.col("score").cast(pl.Float64, strict=False),
        pl.col("taxon_id").cast(pl.Int64, strict=False),
        pl.col("sequence_length").cast(pl.Int64, strict=False),
    ])


orthologs_df = _orthologs_to_frame(orthologs_raw)
print(f"\nDataFrame shape: {orthologs_df.shape[0]:,} rows × {orthologs_df.shape[1]} columns")
orthologs_df.head(10)

### 1.4 Fetch the Hierarchical Orthologous Group (HOG)

A **HOG** is a set of proteins descended from a single ancestral gene at a specified taxonomic level. HOGs are **nested** — a HOG at the Mammalia level is a subset of the HOG at the Euteleostomi level, which is in turn a subset of the root-level HOG. This hierarchy lets us query orthology at variable evolutionary depths.

The `/protein/{id}/hog_derived_orthologs/` endpoint is superseded by two specific endpoints we use below:

1. `/hog/{id}/` — metadata for the HOG containing the query, including the root-level HOG ID.
2. `/hog/{id}/members/` — all protein members of that HOG.

In [ ]:
HOG_META_CACHE = DATA_DIR / f"hog_meta_{SEED_UNIPROT}.json"
HOG_MEMBERS_CACHE = DATA_DIR / f"hog_members_{SEED_UNIPROT}.json"


def fetch_hog(query_id: str, meta_path: Path, members_path: Path) -> tuple[dict, list[dict]]:
    """
    Fetch HOG metadata and member list for a seed protein.

    We query two endpoints in succession:

    1. ``/hog/{query_id}/`` — returns a list describing the HOG at each
       available taxonomic level (deepest first).
    2. ``/hog/{query_id}/members/`` — returns every protein assigned to the
       root-level HOG across all covered genomes.

    Parameters
    ----------
    query_id : str
        Seed protein identifier (UniProt, OMA entry name, etc.). OMA will
        resolve this to the appropriate HOG.
    meta_path, members_path : Path
        Local JSON files for caching each response.

    Returns
    -------
    tuple
        ``(metadata_list, members_list)`` where ``metadata_list`` describes
        the HOG at each taxonomic level and ``members_list`` contains one
        record per member protein.
    """
    # 1) Metadata describing the HOG hierarchy
    if meta_path.exists():
        print(f"Cache hit: {meta_path}")
        hog_meta = json.loads(meta_path.read_text())
    else:
        hog_meta = oma_get(f"hog/{query_id}")
        meta_path.write_text(json.dumps(hog_meta, indent=2))
        print(f"Cached HOG metadata to: {meta_path}")

    # 2) Member list for the HOG
    if members_path.exists():
        print(f"Cache hit: {members_path}")
        members_payload = json.loads(members_path.read_text())
    else:
        members_payload = oma_get(
            f"hog/{query_id}/members", params={"per_page": 10_000}
        )
        members_path.write_text(json.dumps(members_payload, indent=2))
        print(f"Cached HOG members to: {members_path}")

    # The /members/ response is a dict with a "members" list nested inside,
    # but older API versions returned a bare list. Normalise both shapes.
    if isinstance(members_payload, dict):
        members_list = members_payload.get("members", [])
    else:
        members_list = members_payload

    return hog_meta, members_list


hog_meta, hog_members_raw = fetch_hog(SEED_UNIPROT, HOG_META_CACHE, HOG_MEMBERS_CACHE)

# Pretty-print the HOG hierarchy levels returned for this protein
print(f"\nHOG hierarchy levels for UniProt {SEED_UNIPROT}")
print("=" * 60)
# hog_meta is typically a list of dicts: [{level: ..., hog_id: ..., ...}, ...]
meta_records = hog_meta if isinstance(hog_meta, list) else [hog_meta]
for level in meta_records[:15]:
    lvl = level.get("level", "?")
    hid = level.get("hog_id") or level.get("id", "?")
    print(f"  level={lvl:<25} hog_id={hid}")

print(f"\nRetrieved {len(hog_members_raw):,} HOG members")


# ── Flatten HOG members into a Polars DataFrame ─────────────────────────────
def _hog_members_to_frame(records: list[dict]) -> pl.DataFrame:
    """
    Convert raw HOG member records into a Polars DataFrame.

    Parameters
    ----------
    records : list of dict
        Member records as returned by ``/hog/{id}/members/``.

    Returns
    -------
    pl.DataFrame
        Columns: ``omaid``, ``canonicalid``, ``entry_nr``, ``species``,
        ``taxon_id``, ``sequence_length``.
    """
    rows = []
    for r in records:
        sp = r.get("species") or {}
        rows.append({
            "omaid":           r.get("omaid"),
            "canonicalid":     r.get("canonicalid"),
            "entry_nr":        r.get("entry_nr"),
            "species":         sp.get("species") if isinstance(sp, dict) else None,
            "taxon_id":        sp.get("taxon_id") if isinstance(sp, dict) else None,
            "sequence_length": r.get("sequence_length"),
        })

    return pl.DataFrame(rows).with_columns([
        pl.col("taxon_id").cast(pl.Int64, strict=False),
        pl.col("sequence_length").cast(pl.Int64, strict=False),
    ])


hog_members_df = _hog_members_to_frame(hog_members_raw)
print(f"HOG members DataFrame: {hog_members_df.shape[0]:,} rows × {hog_members_df.shape[1]} columns")
hog_members_df.head(10)

### 1.5 Persist Parsed DataFrames as Parquet

Parquet is a compact columnar format that preserves Polars dtypes on round-trip — much faster than re-parsing the JSON caches and a better format for sharing with downstream notebooks.

In [ ]:
ORTH_PARQUET = DATA_DIR / f"orthologs_{SEED_UNIPROT}.parquet"
HOG_PARQUET = DATA_DIR / f"hog_members_{SEED_UNIPROT}.parquet"

# Write both frames to disk. Parquet supports categorical dtypes natively.
orthologs_df.write_parquet(ORTH_PARQUET)
hog_members_df.write_parquet(HOG_PARQUET)

# Verify round-trip readability and file sizes
orth_roundtrip = pl.read_parquet(ORTH_PARQUET)
hog_roundtrip = pl.read_parquet(HOG_PARQUET)

print("Saved Polars DataFrames:")
print(f"  {ORTH_PARQUET}  ({ORTH_PARQUET.stat().st_size / 1024:.1f} KB, "
      f"{orth_roundtrip.shape[0]:,} rows)")
print(f"  {HOG_PARQUET}   ({HOG_PARQUET.stat().st_size / 1024:.1f} KB, "
      f"{hog_roundtrip.shape[0]:,} rows)")

# Quick sanity summary — count orthologs grouped by relationship type
print("\nOrtholog cardinality breakdown:")
print(
    orthologs_df
    .group_by("rel_type")
    .agg(pl.len().alias("n_orthologs"))
    .sort("n_orthologs", descending=True)
)